In [88]:
# book = '/Users/sv/Downloads/ai_warface.epub'
# import zipfile, os

# with zipfile.ZipFile(book, "r") as z:
#     z.extractall("epub_contents")

In [89]:
### CONSTANTS
from dotenv import load_dotenv
load_dotenv()
import zipfile
import os
from openai import OpenAI, AsyncOpenAI
import nest_asyncio
nest_asyncio.apply()
import re
import subprocess
import time
from bs4 import BeautifulSoup 
import requests
book = '/Users/sv/Downloads/ai_warface.epub'
rapidai_api_key = os.getenv('rapidai_api_key')
openai_api_key = os.getenv('openai_api_key')
exa_api_key = os.getenv('exa_api_key')
groq_api_key = os.getenv('groq_api_key')
anthropic_api_key = os.getenv('anthropic_api_key')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

def parse_response(user_input: str):
    """
    Deconstruct user query to extract book_name, author, and file_type.
    Returns list: [book_name, author, file_type]
    Uses fastest OpenAI model available in 2026: gpt-5.4-nano
    """
    # Initialize client with API key from environment variable
    client = OpenAI(api_key=openai_api_key)
    
    # System prompt with proper formatting - fixed quote escaping
    system_prompt = '''Your job is to deconstruct the given user query and return it as a list. 
    The query should contain book_name, author, and file_type. 
    Your response should be as: [book_name, author, file_type]. 
    Return only that list - no json strings, no thinking tags, etc.
    If no author or file type is provided, use "" as the variable. 
    
    User query: {user_input}
    Your response: '''
    
    # Format the prompt correctly using .format() instead of f-string with replace
    formatted_prompt = system_prompt.format(user_input=user_input)
    
    # Use the fastest generation model: gpt-5.4-nano (based on 2026 benchmarks)
    response = client.responses.create(
        model="gpt-5.4-nano",
        input=formatted_prompt
    )
    
    # Extract and return just the response text
    return response.output_text.strip()

def load_api_key():
    openai_api_key = os.getenv('openai_api_key')
    rapidai_api_key = os.getenv('rapidai_api_key')
    return openai_api_key, rapidai_api_key

def retrieve_book(book_name: str, author: str, file_type: list):

    print(f'file type: {type(file_type)}')
    print(f'{file_type}')
    if not author: 
        author = ""
    
    if isinstance(file_type, str):
        file_type = [file_type] if file_type else []
    file_type = [f for f in file_type if f]

    if len(file_type) > 1: file_type = ','.join(file_type)
    elif len(file_type) == 1: file_type = file_type[0]
    else: file_type = ""

    non_member_sources = {'lgli', 'lgrs', 'nexusstc'}
    
    url = "https://annas-archive-api.p.rapidapi.com/search"

    print(f'using filetype: {file_type}')


    querystring = {"q":f"{book_name}","author":f"{author}","cat":"fiction, nonfiction, comic, magazine, musicalscore, other, unknown","page":"1","ext":f"{file_type}","sort":"mostRelevant","source":"libgenLi, libgenRs"}

    headers = {
	"x-rapidapi-key": f"{rapidai_api_key}",
	"x-rapidapi-host": "annas-archive-api.p.rapidapi.com",
	"Content-Type": "application/json"
    }

    response = requests.get(url, headers=headers, params=querystring)
    books = response.json().get('books', [])


    if len(books): 
        extracted = sorted(
            [
                {
                    'title':  book['title'],
                    'author': book['author'],
                    'md5':    book['md5'],
                    'imgUrl': book['imgUrl'],
                    'year':   book['year'], 
                }
                for book in books
                if non_member_sources.intersection(book.get('sources', []))
            ],
            key=lambda x: x['author']
        )

    return extracted 

def get_download_url(md5): 
    url = "https://annas-archive-api.p.rapidapi.com/download"

    querystring = {"md5":f"{md5}"}

    headers = {
        "x-rapidapi-key": f"{rapidai_api_key}",
        "x-rapidapi-host": "annas-archive-api.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    try: 
        response = requests.get(url, headers=headers, params=querystring)
        response.raise_for_status()

        return response.json()
    except Exception as e: 
        return f'Error received when trying to download the book: {e}\n \
            Please try another title.'

def download_book(download_url, book_name): 
    try: 
        r = requests.get(download_url, stream=True)
        with open(book_name, 'wb') as f: 
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f'Successfully downloaded: {book_name}!')
    except Exception as e: 
        print(f'Error when downloading book from m5: {e}')

def get_epub_contents(book, output_path):
    with zipfile.ZipFile(book, "r") as z:
        z.extractall(output_path)
        return output_path


In [90]:
# chapters_index = '/Users/sv/Desktop/audiobooks/epub_contents/OEBPS/xhtml'
# chapters_dir = os.listdir(chapters_index)
# chapters = [chapter for chapter in chapters_dir if '_ch' in chapter]
# print(f'chapters: {chapters}')

In [91]:
from bs4 import BeautifulSoup 

# chapter_text = ''
# output_dir = os.getcwd()
# for filename in sorted(chapters):
#     with open(os.path.join(chapters_index, filename)) as file:
#         soup = BeautifulSoup(file, 'html.parser')
#         text = soup.get_text(separator='\n').strip()
#     with open(os.path.join(output_dir, filename.replace('.xhtml', '.txt')), 'w') as out: 
#         out.write(text)

In [92]:
def clean_text(chapter_text): 
    with open(chapter_text, 'r', encoding='utf-8') as file: 
        text = file.read()
        # numbers on their own line!
        # can also just do simple LLM call
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text



In [93]:
import os 
# 1) get number index between ch and .txt 
# 2) clean the text, use that number index as chapter name 
# 3) look through each to see if anything is missing
# chapter_indices = os.listdir(".")
# print(f'chapter indices: {chapter_indices}')
# chapter_indices = sorted([chapter_index.split('.')[0].split('ch')[-1] for chapter_index in chapter_indices])
# chapter_indices = sorted([int(x) for x in chapter_indices if x.isdigit()])

In [94]:
#chapters = [chapter for chapter in os.listdir('.') if chapter.endswith('.txt')]


In [95]:
# sorted_chapter_files = sorted(chapters, key = lambda x: int(x.split('_ch')[1].split('.txt')[0]))
# print(sorted_chapter_files)

In [96]:
def clean_all_chapters(sorted_chapter_files, chapter_indices):
    for index, chapter in zip(chapter_indices, sorted_chapter_files):
        cleaned_text = clean_text(chapter)
        with open(f'chapter_{index}.txt', 'w', encoding='utf-8') as file:
            file.write(cleaned_text)
        os.remove(chapter)



In [97]:
#clean_all_chapters(sorted_chapter_files, chapter_indices = chapter_indices)

In [98]:
import os
def get_char_count(chapters): 
    for chapter in chapters: 
        chapter_text = open(chapter, 'r', encoding = 'utf-8').read()
        print(len(chapter_text))

# chapters = sorted([chapter for chapter in os.listdir('.') if chapter.endswith('.txt')], 
#     key = lambda x: int(x.replace('chapter_', '').replace('.txt', '')))
# get_char_count(chapters)

In [99]:
def trim_chapter(chapter: str, max_chunk_size: int = 4096) -> list[str]:
    words = chapter.split()
    chunks = []
    current_chunk = []
    current_length = 0 

    for word in words: 
        word_length = len(word) + 1 # why is there +1 
        if current_length + word_length > max_chunk_size: # explain
            chunks.append(' '.join(current_chunk)) # total current chunk - all words 
            current_chunk = [word] # explain -> this resets? 
            current_length = word_length # explain
        else:
            current_chunk.append(word) # adding words if under max
            current_length += word_length # adding word length - this is 4096

    if current_chunk: # when does the loop exit? -> if did not go over 4096, exists in void
        chunks.append(' '.join(current_chunk))

    return chunks

In [100]:
# mk audio dir for each chapter? 
def trim_all_chapter(chapter_text): 
    chapter_chunks = trim_chapter(chapter_text)
    return chapter_chunks 

In [101]:
#os.makedirs("audiobook", exist_ok=True)

In [102]:
# ordered_chapter_texts = sorted([chapter for chapter in os.listdir('.') if chapter.endswith('.txt')],
#     key = lambda x: int(x.replace('chapter_', '').replace('.txt', '')))
# ordered_chapter_indices = sorted([int(chapter.replace('chapter_', '').replace('.txt', '')) for chapter in ordered_chapter_texts])
# print(ordered_chapter_indices)

In [103]:
import time
def chapter_to_audio(chapter_indices, chapters_texts): 
    start_time = time.time()
    for index, chapter in zip(chapter_indices, chapters_texts): 
        chapter_dir = f'audiobook/chapter_{index}'
        os.makedirs(chapter_dir, exist_ok=True)
        chapter_text = open(chapter, 'r', encoding='utf-8').read()
        chapter_chunks = trim_all_chapter(chapter_text)
        for chunk_index, chunk in enumerate(chapter_chunks): 
            response = client.audio.speech.create(
                model = 'tts-1', 
                voice = 'alloy', 
                input = chunk
            )
            response.stream_to_file(f'{chapter_dir}/chunk_{chunk_index}.mp3')
    print(F'one chapter took: {time.time() - start_time}')

In [104]:
#chapter_to_audio(chapter_indices[3:], ordered_chapter_texts[3:])

In [105]:
# try to generate entire audiobook at once - 
# everything should be async
# 1.5 mins across everything
# everything should be async - will be bottlenecked by AI generation time/costs
import time

async def generate_audiobook(
    chapter_indices: list[int], 
    chapter_texts: list[str], ) -> None: 
    start_time = time.time()
    for index, chapter in zip(chapter_indices, chapter_texts): 
        chapter_dir = f'audiobook/chapter_{index}'
        os.makedirs(chapter_dir, exist_ok = True) 
        chapter_text = open(chapter, 'r', encoding='utf-8').read()
        chapter_chunks = trim_all_chapter(chapter_text)

        all_tasks = [
            _save_chunk(chapter_dir, chunk_index, chunk)
            for chunk_index, chunk in enumerate(chapter_chunks)
        ]
        try: 
            await asyncio.gather(*all_tasks)
            print(f"Chapter {index} — all {len(chapter_chunks)} chunk(s) done in one batch.")
        except Exception as e:
            print(f"Chapter {index} — full batch failed ({e}), falling back to batches of 10.")
            for batch_start in range(0, len(chapter_chunks), 10):
                batch = [
                    _save_chunk(chapter_dir, chunk_index, chunk)
                    for chunk_index, chunk in enumerate(
                        chapter_chunks[batch_start : batch_start + 10],
                        start=batch_start,
                    )
                ]
                await asyncio.gather(*batch)
                print(f"Chapter {index} — fallback batch {batch_start}–{batch_start + len(batch) - 1} done.")

        print(f"Chapter {index} complete — {len(chapter_chunks)} chunk(s) total.")
        print(f'Chapter takes {time.time() - start_time} to generate the entire chapter')

async def _save_chunk(chapter_dir: str, chunk_index: int, chunk: str) -> None:
    """Generate and save a single audio chunk."""
    response = await client.audio.speech.create(
        model="tts-1",
        voice="alloy",
        input=chunk,
    )
    output_path = f"{chapter_dir}/chunk_{chunk_index}.mp3"
    response.stream_to_file(output_path)

async def main() -> None:
    await generate_audiobook(ordered_chapter_indices[5:], ordered_chapter_texts[5:])




In [106]:
import os
import re
import sys
import subprocess
import tempfile


def find_chapters(root):
    """
    Scan root for subdirectories matching 'chapter_<N>' (case-insensitive).
    Returns a sorted list of (chapter_num, chapter_dir_path).
    """
    chapter_pattern = re.compile(r"^chapter_(\d+)$", re.IGNORECASE)
    chapters = []

    for entry in os.scandir(root):
        if not entry.is_dir():
            continue
        m = chapter_pattern.match(entry.name)
        if m:
            chapters.append((int(m.group(1)), entry.path))

    return sorted(chapters, key=lambda x: x[0])

In [107]:
import subprocess
# cur_dir = os.getcwd()
# audiobook_dir = cur_dir + '/audiobook'
# os.makedirs('complete_audiobook', exist_ok=True)
# chapters = sorted(os.listdir(audiobook_dir),key=lambda x: int(re.search(r'\d+', x).group()))
# print(f'chapters: {chapters}')
# final_output_path = os.getcwd() + '/final_audiobook.mp3'
# chapter_audiobooks_folder = os.path.join(os.getcwd(), 'complete_audiobook')


In [108]:
def create_chapter_audiobooks(chapters): 
    for chapter in chapters: 
        chunks = [chunk for chunk in os.listdir(os.path.join(audiobook_dir, chapter))]
        sorted_chunks = sorted(chunks, key = lambda x: int(re.search(r'\d+', x).group()))
        final_chunks = [os.path.join(audiobook_dir, chapter, sorted_chunk) for sorted_chunk in sorted_chunks]
        concat_str = '|'.join(final_chunks); print(f'concat string: {concat_str}')
        chapter_output_path = os.path.join(f"{chapter_audiobooks_folder}/{chapter}_audiobook.mp3")
        command = ['ffmpeg', '-i', f'concat:{concat_str}','-acodec','copy',chapter_output_path]
        subprocess.run(command)

In [109]:
#create_chapter_audiobooks(chapters)

In [110]:
def merge_all_chapters_into_final_book(audiobook_chapters): 
    cur_audiobooks = [book for book in os.listdir(audiobook_chapters)]
    sorted_audiobooks = sorted(cur_audiobooks, key = lambda x: int(re.search(r'\d+', x).group()))
    final_sorted_audiobooks = [os.path.join(chapter_audiobooks_folder, audiobook) for audiobook in \
        sorted_audiobooks]
    concat_str = '|'.join(final_sorted_audiobooks)
    print(f'concat_str: {concat_str}')
    final_audiobook_output_path = os.getcwd() + '/final_audiobook.mp3'
    command = ['ffmpeg', '-i', f'concat:{concat_str}','-acodec','copy',final_audiobook_output_path]
    subprocess.run(command)

In [111]:
#merge_all_chapters_into_final_book(chapter_audiobooks_folder)

In [112]:
# existing textbooks 
# existing books 
# article compilations -> APIs* -> Audiobook text -> LLMs (rearrange)
# RAG over existing data(?)*
# podcasts
# start of chapter 1, end of chapter 1 

In [113]:
import datetime 
from exa_py import Exa
from dotenv import load_dotenv 
load_dotenv()
import os 

exa = Exa(api_key=os.getenv('exa_api_key'))



In [114]:
from datetime import date

def make_exa_call(
    query: str = "", 
    num_results: int = 100,  
    start_published_date="2025-09-01",
    end_published_date=str(datetime.date.today()),
    _type: str = 'instant'
):
    # Switch to search_and_contents to retrieve page data
    results = exa.search_and_contents(
        query=query, 
        num_results=num_results,
        start_published_date=start_published_date,
        end_published_date=end_published_date,
        type=_type,
        summary = True,
        text = True
        
    )

    return results

In [115]:
import datetime
from datetime import date

def make_exa_call(
    query: str = "", 
    num_results: int = 30,  
    start_published_date="2025-09-01",
    end_published_date=str(datetime.date.today()),
    _type: str = 'instant'
    #_type: str = 'deep'
):
    # Switch to search_and_contents to retrieve page data
    results = exa.search_and_contents(
        query=query, 
        num_results=num_results,
        start_published_date=start_published_date,
        end_published_date=end_published_date,
        type=_type,
        summary = True,
        text = True
        
    )

    return results

In [116]:
start_time = time.time()
response = make_exa_call(query="How do defense companies get started in Canada?")
print(f'took: {time.time() - start_time}')

took: 5.813604116439819


In [117]:
texts = [result.text for result in response.results]
texts[0]
#with open('sample.txt', 'w', encoding='utf-8') as file: file.write(texts[0])

'How to become a supplier in the defence supply chain \n\n# How to become a supplier in the defence supply chain\n\n8-minute read\n\nPublished January 23, 2026\n\nShare\n\nThe Canadian government has announced it will make an unprecedented series of investments in the defence industry. These investments are driven by the need to bolster national sovereignty, address geopolitical challenges and adhere to Canada’s commitments to NATO.\n\nThe aim is to increase Canadian military spending to 2% of GDP by 2025-2026 and to 5% by 2035. The 2025 federal budget earmarked some $85 billion over five years for defence, including $9.3 billion in 2025-2026. Over the next 10 years, cumulative spending could exceed $1.2 trillion.\n\nThese investments are sure to create business opportunities for Canadian small businesses. More than half (54%) of all expenditures by businesses in the defence supply chain stay in Canada, [according to ISED Canada](https://ised-isde.canada.ca/site/aerospace-defence/en/st

In [118]:
# total_tokens = 0 
# for text in texts: 
#     token_count = token_counter(system_prompt = system_prompt, texts = text)
#     total_tokens += token_count

# print(f'total_tokens: {total_tokens}')

In [119]:
# token_count = token_counter(system_prompt = "", texts = texts[0])
# print(token_count)

In [120]:
from datetime import date

def make_exa_call_summary(
    query: str = "", 
    num_results: int = 30,  
    start_published_date="2025-09-01",
    end_published_date=str(datetime.date.today()),
    _type: str = 'instant'
):
    # Switch to search_and_contents to retrieve page data
    results = exa.search_and_contents(
        query=query, 
        num_results=num_results,
        start_published_date=start_published_date,
        end_published_date=end_published_date,
        type=_type,
        summary = True
        
    )

    return results



In [121]:
#response = make_exa_call_summary(query="How do defense companies get started in Canada?", num_results=)

In [122]:
# texts = [result.text for result in response.results]
# print(len(texts))

In [123]:
from openai import OpenAI
from dotenv import load_dotenv 
load_dotenv()
openai_api_key = os.getenv('openai_api_key')

def organize_text(texts: list[str] = None):
    '''What do you need this AI to do? -> '''
    if not texts: return ""
    system_prompt = '''You are an AI tasked at taking different corpuses of text and arranging 
    them in a way that is linearly relevant. Do not omit information that is important (random details)
    or facts is okay to omit. Just arrange the text so that it is linear/semantically grouped by 
    similarity. You will receive a list of strings as input - return one giant string as your response, 
    that is the the text arranged. Return only the arranged text as your response - no json tags, no thinking strings, etc. 
    '''
    formatted_texts = '\n'.join(f"-{t}" for t in texts)
    user_message = f"<text>\n{formatted_texts}\n</text>"
    client = OpenAI(api_key = openai_api_key)
    response = client.responses.create(
        model="gpt-5-nano",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
    )
    print(response)

    return response.output_text

# either use summaries (and more volume), less results (), more results no cleaning()



In [124]:
# start_time = time.time()
# text = organize_text(texts)
# token_count = token_counter(system_prompt = system_prompt, texts = text)
# print(f'time took: {time.time() - start_time}')
# print(f'total cleaned tokens: {token_count}')

In [ ]:
import tiktoken

def token_counter(texts, system_prompt):
    enc = tiktoken.encoding_for_model("gpt-4o")  # or "gpt-4", "gpt-3.5-turbo"
    
    tokens = enc.encode(system_prompt) + enc.encode(texts)
    return len(tokens)

In [ ]:
from groq import Groq
groq_api_key = os.getenv('groq_api_key')
system_prompt = '''You are an AI tasked at taking different corpuses of text and arranging 
    them in a way that is linearly relevant. Do not omit information that is important (random details)
    or facts is okay to omit. Just arrange the text so that it is linear/semantically grouped by 
    similarity. You will receive a list of strings as input - return one giant string as your response, 
    that is the the text arranged. Return only the arranged text as your response - no json tags, no thinking strings, etc. 
    '''

# formatted_texts = '\n'.join(f"-{t}" for t in texts)
# token_count = token_counter(formatted_texts, system_prompt)
# print(token_count)

12015


In [125]:

def organize_text_groq(texts):
    if not texts: return "You need to supply texts of articles"
    
    formatted_texts = '\n'.join(f"-{t}" for t in texts)
    user_message = f"<text>\n{formatted_texts}\n</text>"
    client = Groq(api_key = groq_api_key)
    completion = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[
      {
        "role": "system",
        "content": f"{system_prompt}"
      },
      {
        "role": "user",
        "content": f"{user_message}"
      }
    ],
        temperature=0.2,
        max_completion_tokens=4096,
        top_p=0.95,
        reasoning_effort="default"
    )

    return completion.choices[0].message.content


In [126]:
import time 
from dotenv import load_dotenv
load_dotenv()
# start_time = time.time()
# cleaned_texts = organize_text_groq(texts)
# print(f'cleaned_texts: {cleaned_texts}') # 
# print(f'time took: {time.time() - start_time}')

True

In [ ]:
# response = make_exa_call(query = "get me all articles for starting a defense company in Canada")
# total_tokens = 0



total_tokens: 117219


In [ ]:
article_indices = range(len(texts))
from dotenv import load_dotenv 
load_dotenv()
def article_to_audio(article_indices, article_texts): 
    start_time = time.time()
    for index, article in zip(article_indices, article_texts): 
        article_dir = f'articles/article_number_{index}'
        os.makedirs(article_dir, exist_ok=True)
        chapter_chunks = trim_all_chapter(article) 
        for chunk_index, chunk in enumerate(chapter_chunks): 
            client = OpenAI(api_key = "nice_try_big_boi")
            response = client.audio.speech.create(
                model = 'tts-1', 
                voice = 'alloy', 
                input = chunk
            )
            response.stream_to_file(f'{article_dir}/chunk_{chunk_index}.mp3')
    print(F'one chapter took: {time.time() - start_time}')





In [129]:
# article_to_audio(article_indices, texts)

In [130]:
import asyncio 
client = AsyncOpenAI(api_key = openai_api_key)

async def clean_article(t: str) -> str:
    '''What do you need this AI to do? -> '''
    if not texts: return ""
    system_prompt = '''You are an AI tasked at cleaning out the articles it receives. Cleaning means removing all non-textual information:
    remove all URL links, all promotional stuff, etc. Anything that is not the actual article. 
    '''
    
    response = await client.responses.create(
        model="gpt-5-nano",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": t}
        ],
    ) 
    return response.output_text




In [131]:
# start_time = time.time()
# cleaned_texts = await asyncio.gather(*[clean_article(t)  for t in texts])
# print(f'cleaned_texts: {cleaned_texts}')
# print(f'{time.time() - start_time}')

In [132]:
from groq import AsyncGroq 

async def async_groq_call(t: str) -> str: 
    client = AsyncGroq(api_key = os.getenv('groq_api_key'))
    chat_completion = await client.chat.completions.create(
        messages=[
            {"role": "system", "content":"""You are an AI tasked at cleaning out the articles it receives. Cleaning means removing all non-textual information:
    remove all URL links, all promotional stuff, etc. Anything that is not the actual article."""},
            {"role": "user", "content": t}
        ],
        model = 'meta-llama/llama-4-scout-17b-16e-instruct',
        temperature = 0.1,
        max_completion_tokens = 8192, 
        top_p = 0.95, 
        stop = None, 
        stream = False,


    )
    return chat_completion.choices[0].message.content

In [133]:
from groq import Groq

def sync_groq_call(t: str) -> str: 
    client = Groq(api_key = os.getenv('groq_api_key'))
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content":"""You are an AI tasked at cleaning out the articles it receives. Cleaning means removing all non-textual information:
    remove all URL links, all promotional stuff, etc. Anything that is not the actual article."""},
            {"role": "user", "content": t}
        ],
        model = 'meta-llama/llama-4-scout-17b-16e-instruct',
        temperature = 0.1,
        max_completion_tokens = 8192, 
        top_p = 0.95, 
        stop = None, 
        stream = False,


    )
    return chat_completion.choices[0].message.content



In [134]:
# sync_groq_call(texts[0])

In [135]:
client = OpenAI(api_key = openai_api_key)

def sync_api_call(t: str) -> str:
    '''What do you need this AI to do? -> '''
    if not texts: return ""
    system_prompt = '''You are an AI tasked at cleaning out the articles it receives. Cleaning means removing all non-textual information:
    remove all URL links, all promotional stuff, etc. Anything that is not the actual article. 
    '''
    
    response = client.responses.create(
        model="gpt-5-nano",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": t}
        ],
    ) 
    return response.output_text

# start_time = time.time()
# sync_api_call(texts[0])
# print(f'{time.time() - start_time}')

In [136]:
# start_time = time.time()
# cleaned_texts = await asyncio.gather(*[async_groq_call(t) for t in texts])
# print(f'time took: {time.time() - start_time}')

In [ ]:
import asyncio 
client = OpenAI(api_key = openai_api_key)
def clean_article(t: str) -> str:
    if not texts: return ""
    cleaning_prompt = '''You are an AI tasked at cleaning out the articles it receives. Cleaning means removing all non-textual information:
    remove all URL links, all promotional stuff, etc. Anything that is not the actual article. Return the exact words 
    that are not relevant to the main article (anything promotional). Return only those words - everything else that is 
    relevant to the main content, do not include in your response'''
    response = client.responses.create(
        model="gpt-5-nano",
        input=[
            {"role": "system", "content": cleaning_prompt},
            {"role": "user", "content": t}
        ],
    ) 
    return response.output_text

In [ ]:
# cleaned_texts = clean_article(texts[0])

In [137]:
import asyncio
from openai import AsyncOpenAI
from dotenv import load_dotenv
load_dotenv()

# Initialize the async client
client = AsyncOpenAI(api_key = os.getenv('openai_api_key'))

cleaning_prompt = '''You are an AI tasked at cleaning out the articles it receives. Cleaning means removing all non-textual information:
    remove all URL links, all promotional stuff, etc. Anything that is not the actual article. Return the exact words 
    that are not relevant to the main article (anything promotional). Return only those words - everything else that is 
    relevant to the main content, do not include in your response'''

async def openai_gen(text):
    # Await the text generation response
    response = await client.responses.create(
        instructions = cleaning_prompt, 
        model="gpt-5.4-nano",
        input=text
    )
    return response.output_text

In [138]:
# 34 seconds for each call 
# cleaned_texts = await asyncio.gather(*[openai_gen(t) for t in texts[:50]])

In [139]:
# textrank, TFIDF, extracing attention weights, Gradient-Based saliency, semantic similarity
# BERTopic* 
with open('sample.txt', 'w', encoding='utf-8') as file: file.write(texts[0])

In [140]:
cleaning_prompt = '''You are an AI tasked at cleaning out the articles it receives. Cleaning means removing all non-textual information:
    remove all URL links, all promotional stuff, etc. Anything that is not the actual article. Return the exact words 
    that are not relevant to the main article (anything promotional). Return only those words - everything else that is 
    relevant to the main content, do not include in your response'''
# 2100 with cleaning prompt
# start_time = time.time()
# print(token_counter("", clean_article(texts[0])))
# print(f'{time.time() - start_time}')

In [141]:
sentences = open('sample.txt', 'r', encoding='utf-8').read()



In [142]:
cleaned_sentences = [line.strip() for line in sentences.splitlines() if line.strip()]
cleaned_sentences # every sentence gets an embedding. 
# What is better? 1) embedding all sentences vs each article as 1 -> do in practice, then find 
# reason why

['How to become a supplier in the defence supply chain',
 '# How to become a supplier in the defence supply chain',
 '8-minute read',
 'Published January 23, 2026',
 'Share',
 'The Canadian government has announced it will make an unprecedented series of investments in the defence industry. These investments are driven by the need to bolster national sovereignty, address geopolitical challenges and adhere to Canada’s commitments to NATO.',
 'The aim is to increase Canadian military spending to 2% of GDP by 2025-2026 and to 5% by 2035. The 2025 federal budget earmarked some $85 billion over five years for defence, including $9.3 billion in 2025-2026. Over the next 10 years, cumulative spending could exceed $1.2 trillion.',
 'These investments are sure to create business opportunities for Canadian small businesses. More than half (54%) of all expenditures by businesses in the defence supply chain stay in Canada, [according to ISED Canada](https://ised-isde.canada.ca/site/aerospace-defenc

In [164]:
import os 
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(api_key = os.getenv('OPENAI_API_KEY'))
print(client)
response = client.embeddings.create(
    input = cleaned_sentences, 
    model = 'text-embedding-3-large'
)

In [165]:
embeddings = [item.embedding for item in response.data] # should be 1 embedding per sentence 

In [170]:
import numpy as np
embeddings_matrix_sentences = np.array(embeddings)
print(embeddings_matrix_sentences.shape)

(50, 3072)


In [171]:
from sklearn.metrics.pairwise import cosine_similarity 
# getting the mean of all embeddings vs getting one embedding for entire document
centroid = embeddings_matrix_sentences.mean(axis=0)
print(f'centroid: {np.array(centroid).shape}')
scores = cosine_similarity([centroid], embeddings_matrix_sentences)[0]
print(f'Totally entries: {len(scores)}')


centroid: (3072,)
Totally entries: 50


In [147]:
indices = np.where((scores > 0.25) & (scores < 0.56))
dirt = np.array(cleaned_sentences)[indices]
dirt
# TFIDF

array(['8-minute read',
       'The aim is to increase Canadian military spending to 2% of GDP by 2025-2026 and to 5% by 2035. The 2025 federal budget earmarked some $85 billion over five years for defence, including $9.3 billion in 2025-2026. Over the next 10 years, cumulative spending could exceed $1.2 trillion.',
       'Original equipment manufacturers (OEMs) are already seeing a rise in orders. Their suppliers need to boost their production to meet growing demand. Requirements are also changing, and those that were previously informal are being formalized.',
       'However, we also advise caution. Although there may be sizable opportunities, not all businesses are ready to enter this cutting-edge industry.',
       '### Plan well before diving in',
       '## The growing importance of information security',
       'If you want to start working in this field, you’ll need to analyze your systems to determine your situation and develop an action plan for implementing a cybersecurity

In [148]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import nltk
nltk.download('punkt_tab')

def remove_irrelevant_sentences(doc, threshold=0.1):
    """
    Remove low-relevance sentences from a document.
    
    Args:
        doc: a single large string
        threshold: sentences with mean TF-IDF below this are removed
    
    Returns:
        cleaned document as a string
    """
    sentences = nltk.sent_tokenize(doc)
    
    # Fit TF-IDF treating each sentence as its own document
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(sentences)
    
    # Score each sentence by the mean TF-IDF of its words
    sentence_scores = np.array(tfidf_matrix.mean(axis=1)).flatten()
    
    # Keep sentences above threshold
    kept_sentences = [s for s, score in zip(sentences, sentence_scores) if score > threshold]
    
    return ' '.join(kept_sentences)

[nltk_data] Downloading package punkt_tab to /Users/sv/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [149]:
cleaned_doc = remove_irrelevant_sentences(sentences, threshold=0.5)
cleaned_doc

''

In [ ]:
# convert each article to sentences 
# function for embedding each doc 
all_sentences = [line.strip() for text in texts for line in text.splitlines() if line.strip()]
# # on avg - 58 * 30
# len(all_sentences)

In [151]:
# mid = len(all_sentences) // 2

# all_sentences_first = all_sentences[:mid]
# all_sentences_second = all_sentences[mid:]

In [152]:
# all_sentences_response_first = client.embeddings.create(
#     input = all_sentences_first, 
#     model = 'text-embedding-3-large'
# )
# first_embeddings = [item.embedding for item in all_sentences_response_first.data]
# len(first_embeddings)

In [153]:
# np.array(first_embeddings).shape

In [154]:
# all_sentences_response_second = client.embeddings.create(
#     input = all_sentences_second, 
#     model = 'text-embedding-3-large'
# )

# second_embeddings = [item.embedding for item in all_sentences_response_second.data]
# len(second_embeddings)

In [155]:
# combined = np.concat((first_embeddings, second_embeddings), axis=0)
# print(np.array(combined).shape)

In [156]:
# # 1) mean of all sentences - centroid for each sentence 
# from sklearn.metrics.pairwise import cosine_similarity 
# # getting the mean of all embeddings vs getting one embedding for entire document
# embeddings_matrix_all = np.array(combined)
# centroid = embeddings_matrix_all.mean(axis=0)
# print(f'centroid: {np.array(centroid).shape}')
# scores = cosine_similarity([centroid], embeddings_matrix_all)[0]
# # use to index the first article -> get the scores for the first article 


In [157]:
# print(scores[:50])

In [172]:
# got mean centroid of all articles 
# goal: remove unnecessary sentences
# what you did: embedded each sentence, got the centroid of each sentence (mean of all documents)
# then calculated off of that
# TFIDF of one, TFIDF of all 
# contrastive sentence scoring, lexical + positional features as prior -> (named entity density)
# get one embedding for all articles vs centroid embedding for all sentences 

def embed_all_articles(articles, client, model='text-embedding-3-large'):
    response = client.embeddings.create(input=articles, model=model)
    all_embeddings = np.array([item.embedding for item in response.data])  # (30, 3072)
    return np.mean(all_embeddings, axis=0)  # already a numpy array

embedding_matrix_all = embed_all_articles(texts, client)
print(type(embedding_matrix_all))

<class 'numpy.ndarray'>


In [173]:
np.array(embedding_matrix_all).shape

(3072,)

In [174]:
# def embed_all_articles(articles, client, model='text-embedding-3-large'):
#     response = client.embeddings.create(input=articles, model=model)
#     all_embeddings = np.array([item.embedding for item in response.data])  # (30, 3072)
#     return np.mean(all_embeddings, axis=0)  # already a numpy array

In [175]:
# mid = len(all_sentences) // 2; 
# first_half = all_sentences[:mid]
# second_half = all_sentences[mid:]
# print(len(first_half))

# first_sentence_embeddings = embed_all_articles(first_half, client)
# second_sentence_embeddings = embed_all_articles(second_half, client)

In [176]:
# combined = np.concat((first_sentence_embeddings, second_sentence_embeddings), axis = 0)
# print(np.array(combined.shape))

In [206]:
# 30 articles 
from sklearn.metrics.pairwise import cosine_similarity 
# print(embedding_matrix_all.shape, embeddings_matrix_sentences.shape)
scores = cosine_similarity(embeddings_matrix_sentences, [embedding_matrix_all])
indices = np.where(scores < 0.25)

In [217]:
bad_indices = [2, 3, 4, 49]
[cleaned_sentences[i] for i in bad_indices]

['8-minute read',
 'Published January 23, 2026',
 'Share',
 '[Feel free to contact us](http://bdc.ca/en/industries/defence) if you think we can provide support or assistance.']

In [218]:
'''embedding all articles into one embedding; getting the centroid (mean) of all articles into 
one embedding -> did not filter all sentences. try IFDTF'''

'embedding all articles into one embedding; getting the centroid (mean) of all articles into \none embedding -> did not filter all sentences. try IFDTF'

In [216]:
indices

(array([ 4,  9, 19, 22, 28, 31]), array([0, 0, 0, 0, 0, 0]))

In [ ]:
for i, sentence in enumerate(cleaned_sentences): 
    print(f'{i}: \t {sentence}')

# indices: 2, 3, 4, 49

0: 	 How to become a supplier in the defence supply chain
1: 	 # How to become a supplier in the defence supply chain
2: 	 8-minute read
3: 	 Published January 23, 2026
4: 	 Share
5: 	 The Canadian government has announced it will make an unprecedented series of investments in the defence industry. These investments are driven by the need to bolster national sovereignty, address geopolitical challenges and adhere to Canada’s commitments to NATO.
6: 	 The aim is to increase Canadian military spending to 2% of GDP by 2025-2026 and to 5% by 2035. The 2025 federal budget earmarked some $85 billion over five years for defence, including $9.3 billion in 2025-2026. Over the next 10 years, cumulative spending could exceed $1.2 trillion.
7: 	 These investments are sure to create business opportunities for Canadian small businesses. More than half (54%) of all expenditures by businesses in the defence supply chain stay in Canada, [according to ISED Canada](https://ised-isde.canada.ca/site/aerosp

In [214]:
int_indices = indices[0].tolist()
print(type(int_indices[0]))
bad_sentences = [cleaned_sentences[i] for i in int_indices]
bad_sentences

<class 'int'>


['Share',
 'Original equipment manufacturers (OEMs) are already seeing a rise in orders. Their suppliers need to boost their production to meet growing demand. Requirements are also changing, and those that were previously informal are being formalized.',
 '### Plan well before diving in',
 'If you decide to take things further, it’s key that you lay the foundation for operational efficiency, quality management and security. ISO 9001 is the basic standard and is a must, regardless of your sector of activity.',
 'If you want to start working in this field, you’ll need to analyze your systems to determine your situation and develop an action plan for implementing a cybersecurity structure.',
 'If you don’t have any contracts, it may be too soon to seek CPCSC certification. However, it might make sense to implement foundational structures. ISO 27001 certification, for example, will help you make headway and prevent major surprises when the time comes to obtain more advanced certifications

In [231]:
# TFIDF - 30 articles. Want to see if a sentence belongs compared to the rest of the articles 
# Density-based anomaly detection; textrank 
from typing import List 
import re 
import pandas as pd 
import numpy as np 

print(all_sentences)

def tokenize(text: str) -> List[str]: 
    cleaned_text = re.sub(r"[^\w\s]", "", text)
    tokens = cleaned_text.lower().split()
    return tokens 

doc_a = tokenize(sentence_a)
doc_b = tokenize(sentence_b)

print(len(doc_a), len(doc_b))

total_corpus = set(doc_a).union(set(doc_b))
print(len(total_corpus))

word_count_a = dict.fromkeys(total_corpus, 0)
word_count_b = dict.fromkeys(total_corpus, 0)

for word in doc_a: 
    word_count_a[word] += 1 # count of words in each doc 

for word in doc_b: 
    word_count_b[word] += 1 

# total freq of terms in each doc

print(word_count_a, '\n', word_count_b)

pd.set_option('display.max_rows', None)
freq = pd.DataFrame([word_count_a, word_count_b])

def tf(word_counts: dict, document: list[str]) -> dict: 
    tf_dict = {}
    corpus_count = len(document) # total amt of words 
    for word, count in word_counts.items():
        # amt of time word appears in document compared to all words 
        tf_dict[word] = count / float(corpus_count)

    return tf_dict


def idf(word_counts: list[dict[str, int]]) -> dict: 
    idf_dict = {}
    # total num of documents
    N = len(word_counts)

    idf_dict = dict.fromkeys(word_counts[0])

    for word in idf_dict.keys(): 
        # get occurrence of each word in all documents
        idf_dict[word] = sum(doc[word] > 0 for doc in word_counts)

    
    for word, df in idf_dict.items(): 
        idf_dict[word] = np.log10((N + 1.0) / (df + 1.0))

    return idf_dict 


['How to become a supplier in the defence supply chain', '# How to become a supplier in the defence supply chain', '8-minute read', 'Published January 23, 2026', 'Share', 'The Canadian government has announced it will make an unprecedented series of investments in the defence industry. These investments are driven by the need to bolster national sovereignty, address geopolitical challenges and adhere to Canada’s commitments to NATO.', 'The aim is to increase Canadian military spending to 2% of GDP by 2025-2026 and to 5% by 2035. The 2025 federal budget earmarked some $85 billion over five years for defence, including $9.3 billion in 2025-2026. Over the next 10 years, cumulative spending could exceed $1.2 trillion.', 'These investments are sure to create business opportunities for Canadian small businesses. More than half (54%) of all expenditures by businesses in the defence supply chain stay in Canada, [according to ISED Canada](https://ised-isde.canada.ca/site/aerospace-defence/en/st

In [227]:
idfs = idf([word_count_a, word_count_b])
idfs

{'it': np.float64(0.17609125905568124),
 'time': np.float64(0.17609125905568124),
 'your': np.float64(0.17609125905568124),
 'dont': np.float64(0.17609125905568124),
 'enough': np.float64(0.17609125905568124),
 'elses': np.float64(0.17609125905568124),
 'long': np.float64(0.17609125905568124),
 'lived': np.float64(0.17609125905568124),
 'waste': np.float64(0.17609125905568124),
 'limited': np.float64(0.17609125905568124),
 'is': np.float64(0.0),
 'life': np.float64(0.0),
 'well': np.float64(0.17609125905568124),
 'living': np.float64(0.17609125905568124),
 'so': np.float64(0.17609125905568124),
 'if': np.float64(0.17609125905568124),
 'someone': np.float64(0.17609125905568124)}

In [228]:
def tfidf(doc_elements: dict[str, int], idfs: dict[str, int]) -> dict: 
    '''TF * IDF per word given a single word in a single document.''' 
    tfidf_dict = {}
    for word, val in doc_elements.items():
        tfidf_dict[word] = val * idfs[word]
    return tfidf_dict 

In [229]:
tf_a = tf(word_count_a, doc_a)
tf_b = tf(word_count_b, doc_b)

tfidf_a = tfidf(tf_a, idfs)
tfidf_b = tfidf(tf_b, idfs)

document_tfidf = pd.DataFrame([tfidf_a, tfidf_b])
document_tfidf.T



,0,1
it,0.000000,0.014674
time,0.000000,0.014674
your,0.000000,0.014674
dont,0.000000,0.014674
enough,0.025156,0.000000
elses,0.000000,0.014674
long,0.025156,0.000000
lived,0.025156,0.000000
waste,0.000000,0.014674
limited,0.000000,0.014674


In [257]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = texts

# Split into sentences (each sentence is now a "document")
sentences = all_sentences

# Fit TF-IDF where each sentence is treated as a document
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(sentences)

# Compare a query sentence against all sentences
query = cleaned_sentences
query_vec = vectorizer.transform(query)

similarities = cosine_similarity(query_vec, tfidf_matrix)
print(np.array(similarities).shape)  # Score of query vs each sentence

(50, 3636)


In [235]:
bad_indices

[2, 3, 4, 49]

In [236]:
[similarities[i] for i in bad_indices]

[array([0., 0., 1., ..., 0., 0., 0.], shape=(3636,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(3636,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(3636,)),
 array([0.03474982, 0.03474982, 0.        , ..., 0.02219318, 0.06606149,
        0.11789171], shape=(3636,))]

In [258]:
total_sum = 0 
for sentence in similarities: 
    total_sum += sum(sentence)

print(total_sum)

4866.3913948606105


In [ ]:
# 4866, 97 
avg_score = total_sum / len(similarities)
avg_score

np.float64(97.32782789721222)

In [ ]:
first_article = similarities[0] # 0.03 - last sentence is useful 
first_article[9]
# TFIDF seems most promising, but still uncertain. 

np.float64(0.03416142442759597)

In [1]:
regular_tts = '''How to become a supplier in the defence supply chain 

# How to become a supplier in the defence supply chain

8-minute read

Published January 23, 2026

Share

The Canadian government has announced it will make an unprecedented series of investments in the defence industry. These investments are driven by the need to bolster national sovereignty, address geopolitical challenges and adhere to Canada’s commitments to NATO.

The aim is to increase Canadian military spending to 2% of GDP by 2025-2026 and to 5% by 2035. The 2025 federal budget earmarked some $85 billion over five years for defence, including $9.3 billion in 2025-2026. Over the next 10 years, cumulative spending could exceed $1.2 trillion.

These investments are sure to create business opportunities for Canadian small businesses. More than half (54%) of all expenditures by businesses in the defence supply chain stay in Canada, [according to ISED Canada](https://ised-isde.canada.ca/site/aerospace-defence/en/state-canadas-defence-industry). Increased spending should therefore lead to an increase in orders for these businesses, especially as the government commits to focusing on Canadian suppliers.

In particular, new equipment purchases will open up new opportunities for small businesses in the aerospace, shipbuilding and automobile manufacturing industries. These businesses could also benefit from access to allied nations’ supply chains.

Original equipment manufacturers (OEMs) are already seeing a rise in orders. Their suppliers need to boost their production to meet growing demand. Requirements are also changing, and those that were previously informal are being formalized.

These trends are promising for those interested in entering the industry, particularly at a time when leading industrial companies are seeking to diversify in Canada to better manage this growth.

Without solid cybersecurity structures, your business might not be able to access the industry.

## How to seize opportunities in the defence industry

We’re noticing growing interest in Canada’s defence industry in our daily interactions with entrepreneurs.

However, we also advise caution. Although there may be sizable opportunities, not all businesses are ready to enter this cutting-edge industry.

Defence is a highly regulated field. It calls for specialized certifications and security licences before you can even begin to generate revenue. Requirements for operating in the defence industry generally fall into three categories:

1. Controlled goods and security licencesBusinesses need to register for the [Controlled Goods Program](https://www.canada.ca/en/public-services-procurement/services/industrial-security/controlled-goods.html) to examine, possess or transfer controlled goods in Canada. This program is often one of the first certifications required. You may also need to obtain specialized attestations to comply with various security requirements.
2. CybersecurityBusinesses that bid on certain Canadian government defence contracts or that wish to work in the field must now successfully complete the Canadian Program for [Cyber Security Certification (CPCSC)](https://www.canada.ca/en/public-services-procurement/services/industrial-security/security-requirements-contracting/cyber-security-certification-defence-suppliers-canada.html).While not all businesses in the supply chain need this certification, you may be asked to meet other cybersecurity requirements.For example, [ISO 27001](http://bdc.ca/en/articles-tools/operations/iso-other-certifications/how-iso-certification-help-secure-businesss-information) can serve as a solid foundation for most businesses, whereas the Cybersecurity Maturity Model Certification 2.0 (CMMC 2.0) is often required for defence contracts in the United States. Without solid cybersecurity structures, your business may struggle to access the industry.
3. Quality control systemsDefence contracts often have strict quality requirements, enforced through certifications such as [ISO 9001](http://bdc.ca/en/articles-tools/entrepreneur-toolkit/templates-business-guides/glossary/iso-9001), or AS9100 for businesses in the aerospace industry. Suppliers must meet these requirements to avoid being excluded from the industry.

### Plan well before diving in

In the defence industry, the requirements imposed on suppliers vary according to the type of contract. Tier 1 suppliers, like OEMs, are generally subject to stricter requirements. Tier 2 suppliers, which manufacture parts or key sub-assemblies used by Tier 1 businesses, and Tier 3 suppliers, which provide raw materials, basic components, or specialized services for Tier 2 and sometimes Tier 1 operations, are subject to less stringent regulations.

You should therefore start with an operational and strategic analysis of your business to get a clear picture as to whether your skills match the industry’s requirements. For example, if you’re an automobile supplier specialized in welding, you likely already work with similar welding standards to those required by the defence industry for Tier 3 suppliers. These will give you a solid foundation.

If you decide to take things further, it’s key that you lay the foundation for operational efficiency, quality management and security. ISO 9001 is the basic standard and is a must, regardless of your sector of activity.

If you haven’t yet found customers, it’s a good idea to prioritize implementing these building blocks before proceeding. Some certifications are expensive, but ISO 9001 and cybersecurity measures are prerequisites, regardless of whether you’re aiming to work in the defence industry.

It’s often more difficult to obtain certifications in the defence industry if you don’t already have customers in the field. As these certifications can be very costly, some businesses opt to only certify a portion of their workshop or IT system.

Regardless of your strategy, it’s crucial to have a thorough understanding of your customers’ requirements and to build a solid foundation in terms of quality and cybersecurity from the start.

## The growing importance of information security

Information security is a top priority in the defence industry.

If you want to start working in this field, you’ll need to analyze your systems to determine your situation and develop an action plan for implementing a cybersecurity structure.

Cybersecurity standards aim to regulate the sharing of information within a company. If an OEM orders a part from you, they must know that access to the plans is secure. Develop a management plan to decide who will have access, how you will manufacture the part, and how you will prevent any leaks, such as printing, photos, or plans, leaving the premises.

For most projects, it’s a question of controlling and securing information according to its risk level.

If you don’t have any contracts, it may be too soon to seek CPCSC certification. However, it might make sense to implement foundational structures. ISO 27001 certification, for example, will help you make headway and prevent major surprises when the time comes to obtain more advanced certifications.

### Beware of wasting your resources

We’ve seen clients who had long neglected cybersecurity be forced to spend thousands of dollars updating their systems. This can require changing IT service suppliers, as not all are able to ensure that your systems comply with industry standards.

If they’re aware that a project involves the defence industry, some suppliers will offer oversized systems or inflated assessments. Some consultation firms, which are also resellers, will prioritize their own products over your needs. Calling on an impartial third party, such as BDC, can help avoid this type of situation.

Advanced cybersecurity certifications require more expensive systems, but there are often more affordable solutions. Be sure to take regional variations into account, analyze your needs and develop a roadmap to avoid wasting your resources.

Most business development in the defence sector happens through word-of-mouth. Many businesses develop gradually based on their products’ reputation.

## How to find customers in the defence industry

We suggest that you focus on brand recognition. Most business development in the defence sector happens through word of mouth. Many businesses develop gradually based on their products’ reputation.

Maybe you’ve already successfully delivered a small order? Others offer products that are so specialized that industry players have no choice but to do business with them. In either case, you’ll likely need to rapidly obtain your certifications.

Trade shows and industry events are also a key entry point. These can be international events, such as the International Paris Trade Show for the European aerospace industry, but there are others held in Canada. Major OEMs often attend events for steel, aluminum and polymer producers in order to identify suppliers that meet their needs.

Some businesses try to land new contracts through cold calling, but this isn’t effective and should be avoided. Most contracts in the industry are granted through government calls for tenders, but if you’re new to the industry, certification requirements may keep you from participating in them.

You can’t manage a defence project with a spreadsheet. Be sure you have sufficient systems and the appropriate production capacity. In the defence industry, every detail counts. There’s zero tolerance for mistakes.

## Be strategic

New investments in the defence industry will create business opportunities for small and medium-sized enterprises in Canada. To fully take advantage of them, you need to be well organized.

Industry standards are evolving rapidly and becoming increasingly stringent. Meeting them requires time, effort and investment. These costs must be considered when assessing a business opportunity in the defence industry.

If your business has already worked in this field, less effort will be required. However, we’ve noted that the industry is becoming more professional. Stay informed about the latest developments to avoid missing out.

If your business is new to the industry, it’s important to take the proper steps. You can’t manage a defence project with a spreadsheet. Be sure you have sufficient systems and the appropriate production capacity. In the defence industry, every detail counts. There’s zero tolerance for mistakes.

For businesses that are up for the challenge, numerous opportunities will be available. At BDC, we understand small businesses like yours and are here to guide you every step of the way, providing financing, capital and tailored advice.

[Feel free to contact us](http://bdc.ca/en/industries/defence) if you think we can provide support or assistance.'''

In [ ]:
import pyttsx3 

engine = pyttsx3.init()

engine.setProperty('rate', 180)
engine.setProperty('volume', 1.0)

engine.say(regular_tts)
engine.runAndWait()


In [ ]:
import subprocess 

subprocess.run(['say', regular_tts])

In [ ]:
import asyncio
import edge_tts

async def try_again():
    communicate = edge_tts.Communicate(regular_tts, "en-US-AriaNeural")
    await communicate.save("edge_output.mp3")

await try_again() # voice is more annoying but reads the text better 